# 019 — Data Loaders for RAG
Four real-world data sources every RAG pipeline must handle:
1. **SQLite database** — structured records → Documents
2. **PDF** — research paper (Attention Is All You Need)
3. **YouTube transcript** — video captions → Documents
4. **JSON dataset** — structured Q&A with metadata


## Setup

In [1]:
import os, warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', message=".*LangChain.*")
warnings.filterwarnings('ignore', message=".*allowed_objects.*")
warnings.filterwarnings('ignore', module="langchain.*")
warnings.filterwarnings('ignore', module="langgraph.*")
from dotenv import load_dotenv
load_dotenv(os.path.join(os.getcwd(), '..', '.env'))
GROQ_API_KEY = os.getenv('GROQ_API_KEY')
assert GROQ_API_KEY, "GROQ_API_KEY not found — check ../.env"
print("✓ API key loaded")


✓ API key loaded


In [2]:
import os, time
os.environ["ANONYMIZED_TELEMETRY"] = "False"
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings

llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0, max_retries=2)
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    cache_folder=os.path.join(os.getcwd(), '..', 'datasets', 'models'),
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)
DS_DIR = os.path.join(os.getcwd(), '..', 'datasets')
print("✓ LLM + embeddings ready")
print(f"✓ Datasets dir: {DS_DIR}")


✓ LLM + embeddings ready
✓ Datasets dir: /home/saurabh/projects/personal_project/advance-rag/notebooks/../datasets


---
## 1. SQLite Database Loader
Create a sample `ai_papers` database, then load rows as LangChain `Document` objects.


In [3]:
import sqlite3, json
from langchain.schema import Document

DB_PATH = os.path.join(DS_DIR, 'ai_papers.db')

# ── Create & populate ────────────────────────────────────────────────────
conn = sqlite3.connect(DB_PATH)
cur  = conn.cursor()

cur.execute('''DROP TABLE IF EXISTS ai_papers''')
cur.execute('''
    CREATE TABLE ai_papers (
        id        INTEGER PRIMARY KEY,
        title     TEXT,
        authors   TEXT,
        year      INTEGER,
        venue     TEXT,
        citations INTEGER,
        topic     TEXT,
        abstract  TEXT
    )
''')

papers = [
    (1, "Attention Is All You Need",
     "Vaswani et al.", 2017, "NeurIPS", 80000, "Transformers",
     "We propose a new simple network architecture, the Transformer, based solely on attention "
     "mechanisms, dispensing with recurrence and convolutions entirely."),
    (2, "BERT: Pre-training of Deep Bidirectional Transformers",
     "Devlin et al.", 2019, "NAACL", 55000, "Pre-training",
     "We introduce BERT, which stands for Bidirectional Encoder Representations from Transformers. "
     "BERT is designed to pre-train deep bidirectional representations from unlabelled text."),
    (3, "GPT-3: Language Models are Few-Shot Learners",
     "Brown et al.", 2020, "NeurIPS", 35000, "Large Language Models",
     "We demonstrate that scaling language models greatly improves task-agnostic few-shot performance. "
     "GPT-3 achieves strong results on NLP benchmarks with few or zero examples."),
    (4, "CLIP: Learning Transferable Visual Models from Natural Language",
     "Radford et al.", 2021, "ICML", 18000, "Multimodal",
     "We study the behaviour of CLIP, a neural network trained on image-text pairs. "
     "CLIP can be applied to any visual classification benchmark by providing category names."),
    (5, "Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks",
     "Lewis et al.", 2020, "NeurIPS", 6000, "RAG",
     "We combine pre-trained parametric and non-parametric memory for language generation. "
     "Our RAG models use a dense retriever to retrieve supporting documents."),
    (6, "LangChain: Building Applications with LLMs through Composability",
     "Chase", 2022, "GitHub", 2000, "LLM Frameworks",
     "LangChain is a framework for developing applications powered by language models. "
     "It provides chains, agents, memory, and retrieval primitives."),
    (7, "Self-RAG: Learning to Retrieve, Generate, and Critique",
     "Asai et al.", 2023, "arXiv", 800, "Advanced RAG",
     "We introduce SELF-RAG, a framework that trains a language model to adaptively retrieve, "
     "generate, and critique passages and its own generations using reflection tokens."),
    (8, "LlamaIndex: A Data Framework for LLM Applications",
     "Liu", 2022, "GitHub", 1500, "LLM Frameworks",
     "LlamaIndex provides data connectors, indexes, and query engines to connect custom data "
     "sources to large language models."),
    (9, "FAISS: A Library for Efficient Similarity Search",
     "Johnson et al.", 2021, "IEEE TPAMI", 3000, "Vector Search",
     "We present FAISS, a library that provides highly optimized routines for clustering, "
     "compression, and efficient similarity search of dense vectors."),
    (10, "Chain-of-Thought Prompting Elicits Reasoning in Large Language Models",
     "Wei et al.", 2022, "NeurIPS", 5000, "Prompting",
     "We explore chain-of-thought prompting as a simple mechanism for eliciting reasoning. "
     "A series of intermediate reasoning steps improves complex reasoning on LLMs."),
]

cur.executemany(
    "INSERT INTO ai_papers VALUES (?,?,?,?,?,?,?,?)", papers
)
conn.commit()
conn.close()

print(f"✓ Database created: {DB_PATH}")
print(f"✓ Inserted {len(papers)} papers")


✓ Database created: /home/saurabh/projects/personal_project/advance-rag/notebooks/../datasets/ai_papers.db
✓ Inserted 10 papers


In [4]:
# ── Load from DB as LangChain Documents ─────────────────────────────────
def load_db_as_documents(db_path: str, topic_filter: str = None) -> list[Document]:
    conn = sqlite3.connect(db_path)
    cur  = conn.cursor()
    if topic_filter:
        cur.execute("SELECT * FROM ai_papers WHERE topic = ?", (topic_filter,))
    else:
        cur.execute("SELECT * FROM ai_papers")
    rows = cur.fetchall()
    conn.close()

    docs = []
    for row in rows:
        id_, title, authors, year, venue, citations, topic, abstract = row
        page_content = f"Title: {title}\nAuthors: {authors}\nAbstract: {abstract}"
        metadata = {
            "source": "sqlite",
            "table": "ai_papers",
            "id": id_,
            "title": title,
            "authors": authors,
            "year": year,
            "venue": venue,
            "citations": citations,
            "topic": topic,
        }
        docs.append(Document(page_content=page_content, metadata=metadata))
    return docs

# Load all papers
all_docs = load_db_as_documents(DB_PATH)
print(f"✓ Loaded {len(all_docs)} documents from SQLite")
print()

# Show first document
doc = all_docs[0]
print("── Document 1 ──────────────────────────────")
print(doc.page_content)
print()
print("── Metadata ────────────────────────────────")
for k, v in doc.metadata.items():
    print(f"  {k}: {v}")


✓ Loaded 10 documents from SQLite

── Document 1 ──────────────────────────────
Title: Attention Is All You Need
Authors: Vaswani et al.
Abstract: We propose a new simple network architecture, the Transformer, based solely on attention mechanisms, dispensing with recurrence and convolutions entirely.

── Metadata ────────────────────────────────
  source: sqlite
  table: ai_papers
  id: 1
  title: Attention Is All You Need
  authors: Vaswani et al.
  year: 2017
  venue: NeurIPS
  citations: 80000
  topic: Transformers


In [5]:
# ── Query with filter ────────────────────────────────────────────────────
rag_docs = load_db_as_documents(DB_PATH, topic_filter="RAG")
print(f"✓ RAG papers: {len(rag_docs)}")
for d in rag_docs:
    print(f"  [{d.metadata['year']}] {d.metadata['title']} — citations: {d.metadata['citations']}")

# ── Simple SQL query demo ────────────────────────────────────────────────
conn = sqlite3.connect(DB_PATH)
cur  = conn.cursor()
cur.execute("SELECT topic, COUNT(*), AVG(citations) FROM ai_papers GROUP BY topic ORDER BY AVG(citations) DESC")
print()
print("── Papers by topic (avg citations) ─────────")
print(f"{'Topic':<25} {'Count':>5} {'Avg Citations':>15}")
print("-" * 47)
for topic, count, avg_cit in cur.fetchall():
    print(f"{topic:<25} {count:>5} {avg_cit:>15,.0f}")
conn.close()


✓ RAG papers: 1
  [2020] Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks — citations: 6000

── Papers by topic (avg citations) ─────────
Topic                     Count   Avg Citations
-----------------------------------------------
Transformers                  1          80,000
Pre-training                  1          55,000
Large Language Models         1          35,000
Multimodal                    1          18,000
RAG                           1           6,000
Prompting                     1           5,000
Vector Search                 1           3,000
LLM Frameworks                2           1,750
Advanced RAG                  1             800


---
## 2. PDF Loader
Load the **"Attention Is All You Need"** paper (NeurIPS 2017) from `datasets/`.


In [6]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

PDF_PATH = os.path.join(DS_DIR, 'NIPS-2017-attention-is-all-you-need-Paper.pdf')

# ── Load pages ───────────────────────────────────────────────────────────
loader = PyPDFLoader(PDF_PATH)
pdf_pages = loader.load()

print(f"✓ PDF loaded: {os.path.basename(PDF_PATH)}")
print(f"✓ Total pages: {len(pdf_pages)}")
print()

# Show first page info
first = pdf_pages[0]
print("── Page 1 metadata ─────────────────────────")
for k, v in first.metadata.items():
    print(f"  {k}: {v}")
print()
print("── Page 1 content (first 500 chars) ─────────")
print(first.page_content[:500])


✓ PDF loaded: NIPS-2017-attention-is-all-you-need-Paper.pdf
✓ Total pages: 11

── Page 1 metadata ─────────────────────────
  producer: PyPDF2
  creator: PyPDF
  creationdate: 
  subject: Neural Information Processing Systems http://nips.cc/
  publisher: Curran Associates, Inc.
  language: en-US
  created: 2017
  eventtype: Poster
  description-abstract: The dominant sequence transduction models are based on complex recurrent orconvolutional neural networks in an encoder and decoder configuration. The best performing such models also connect the encoder and decoder through an attentionm echanisms.  We propose a novel, simple network architecture based solely onan attention mechanism, dispensing with recurrence and convolutions entirely.Experiments on two machine translation tasks show these models to be superiorin quality while being more parallelizable and requiring significantly less timeto train. Our single model with 165 million parameters, achieves 27.5 BLEU onEnglish-to-German tr

In [7]:
# ── Show all pages with length ──────────────────────────────────────────
print("── Page lengths ─────────────────────────────")
for i, page in enumerate(pdf_pages):
    print(f"  Page {i+1:2d}: {len(page.page_content):5d} chars")


── Page lengths ─────────────────────────────
  Page  1:  2909 chars
  Page  2:  4251 chars
  Page  3:  1753 chars
  Page  4:  2442 chars
  Page  5:  3194 chars
  Page  6:  3523 chars
  Page  7:  3213 chars
  Page  8:  3313 chars
  Page  9:  2619 chars
  Page 10:  3089 chars
  Page 11:  2325 chars


In [8]:
# ── Split PDF into RAG-ready chunks ─────────────────────────────────────
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100,
    separators=["\n\n", "\n", ". ", " ", ""],
)
pdf_chunks = splitter.split_documents(pdf_pages)

print(f"✓ PDF split into {len(pdf_chunks)} chunks")
print(f"  Avg chunk size: {sum(len(c.page_content) for c in pdf_chunks)//len(pdf_chunks)} chars")
print()

# Sample chunk from middle of paper
mid = len(pdf_chunks) // 2
print(f"── Chunk {mid} (from page {pdf_chunks[mid].metadata.get('page', '?')}) ──")
print(pdf_chunks[mid].page_content[:400])


✓ PDF split into 51 chunks
  Avg chunk size: 690 chars

── Chunk 25 (from page 5) ──
versions produced nearly identical results (see Table 3 row (E)). We chose the sinusoidal version
because it may allow the model to extrapolate to sequence lengths longer than the ones encountered
during training.
4 Why Self-Attention
In this section we compare various aspects of self-attention layers to the recurrent and convolu-
tional layers commonly used for mapping one variable-length sequenc


In [9]:
# ── Ask the LLM about the paper ─────────────────────────────────────────
from langchain_core.prompts import ChatPromptTemplate
import faiss, numpy as np

# Build a tiny in-memory FAISS index over PDF chunks
texts  = [c.page_content for c in pdf_chunks]
vecs   = np.array(embeddings.embed_documents(texts), dtype="float32")
index  = faiss.IndexFlatL2(vecs.shape[1])
index.add(vecs)

def pdf_qa(question: str, k: int = 3) -> str:
    q_vec = np.array(embeddings.embed_query(question), dtype="float32").reshape(1, -1)
    _, idx = index.search(q_vec, k)
    context = "\n\n".join(texts[i] for i in idx[0])
    prompt = ChatPromptTemplate.from_template(
        "Answer using ONLY the context below.\n\nContext:\n{ctx}\n\nQuestion: {q}"
    )
    return llm.invoke(prompt.format_messages(ctx=context, q=question)).content

q1 = "What is the main innovation in the Transformer architecture?"
print(f"Q: {q1}")
print(f"A: {pdf_qa(q1)}")
print()
q2 = "How many attention heads did the authors use in their base model?"
print(f"Q: {q2}")
print(f"A: {pdf_qa(q2)}")


Q: What is the main innovation in the Transformer architecture?


A: The main innovation in the Transformer architecture is the use of stacked self-attention and point-wise, fully connected layers for both the encoder and decoder, which allows for parallel computation of hidden representations for all input and output positions, reducing sequential computation.

Q: How many attention heads did the authors use in their base model?


A: The authors employed h = 8 parallel attention layers, or heads.


---
## 3. YouTube Transcript Loader
Load video captions as Documents. We show the real API pattern and fall back to a
pre-saved transcript when the network is restricted (e.g. CI environments).

**Video:** *But what is a neural network?* — 3Blue1Brown
**Video ID:** `aircAruvnKk`


In [10]:
import json
from langchain.schema import Document

VIDEO_ID  = "aircAruvnKk"
VIDEO_URL = f"https://www.youtube.com/watch?v={VIDEO_ID}"
FALLBACK  = os.path.join(DS_DIR, 'yt_transcript_sample.json')

def fetch_youtube_transcript(video_id: str) -> list[dict]:
    '''Try real API; fall back to local sample if network is restricted.'''
    try:
        from youtube_transcript_api import YouTubeTranscriptApi
        transcript_list = YouTubeTranscriptApi.list_transcripts(video_id)
        transcript = transcript_list.find_transcript(['en'])
        return transcript.fetch()
    except Exception as e:
        print(f"  ⚠ YouTube API unavailable ({type(e).__name__}), loading local fallback.")
        with open(FALLBACK, encoding='utf-8') as f:
            return json.load(f)

raw_transcript = fetch_youtube_transcript(VIDEO_ID)
print(f"✓ Transcript entries: {len(raw_transcript)}")
print()
print("── First 5 entries ──────────────────────────")
for entry in raw_transcript[:5]:
    t = entry['start']
    mins, secs = int(t) // 60, int(t) % 60
    print(f"  [{mins:02d}:{secs:02d}]  {entry['text']}")


  ⚠ YouTube API unavailable (ParseError), loading local fallback.
✓ Transcript entries: 42

── First 5 entries ──────────────────────────
  [00:00]  When you see a photo like this, you can immediately tell that it's a picture of a bumblebee.
  [00:04]  How it is that biological nervous systems can do this so effortlessly is one of the great mysteries of neuroscience.
  [00:08]  But artificial neural networks take a different approach: learning by example.
  [00:12]  A neural network is made of layers of neurons. Each neuron holds a number between zero and one.
  [00:17]  The numbers in the first layer are the inputs — for an image, each pixel maps to one neuron.


In [11]:
# ── Merge entries into timed chunks (~60 second windows) ────────────────
def transcript_to_documents(entries: list[dict], window_sec: float = 60.0,
                             video_id: str = "") -> list[Document]:
    docs, chunk_texts, chunk_start = [], [], 0.0
    for entry in entries:
        if entry['start'] - chunk_start >= window_sec and chunk_texts:
            text = " ".join(chunk_texts)
            docs.append(Document(
                page_content=text,
                metadata={
                    "source": "youtube",
                    "video_id": video_id,
                    "url": f"https://www.youtube.com/watch?v={video_id}&t={int(chunk_start)}s",
                    "start_sec": chunk_start,
                    "end_sec": entry['start'],
                    "duration_sec": entry['start'] - chunk_start,
                }
            ))
            chunk_texts, chunk_start = [], entry['start']
        chunk_texts.append(entry['text'])

    # flush last chunk
    if chunk_texts:
        docs.append(Document(
            page_content=" ".join(chunk_texts),
            metadata={"source": "youtube", "video_id": video_id,
                      "start_sec": chunk_start, "end_sec": entries[-1]['start'],
                      "url": f"https://www.youtube.com/watch?v={video_id}&t={int(chunk_start)}s"}
        ))
    return docs

yt_docs = transcript_to_documents(raw_transcript, window_sec=60.0, video_id=VIDEO_ID)

print(f"✓ YouTube transcript → {len(yt_docs)} timed chunks")
print()
for i, doc in enumerate(yt_docs):
    m = doc.metadata
    start_min = int(m['start_sec']) // 60
    start_sec = int(m['start_sec']) % 60
    print(f"  Chunk {i+1}: [{start_min:02d}:{start_sec:02d}]  {len(doc.page_content)} chars  →  {m['url']}")


✓ YouTube transcript → 4 timed chunks

  Chunk 1: [00:00]  1223 chars  →  https://www.youtube.com/watch?v=aircAruvnKk&t=0s
  Chunk 2: [01:01]  1203 chars  →  https://www.youtube.com/watch?v=aircAruvnKk&t=61s
  Chunk 3: [02:03]  1188 chars  →  https://www.youtube.com/watch?v=aircAruvnKk&t=123s
  Chunk 4: [03:05]  295 chars  →  https://www.youtube.com/watch?v=aircAruvnKk&t=185s


In [12]:
# ── Show one chunk content ───────────────────────────────────────────────
print("── Chunk 1 content ─────────────────────────")
print(yt_docs[0].page_content)
print()
print("── Chunk 1 metadata ────────────────────────")
for k, v in yt_docs[0].metadata.items():
    print(f"  {k}: {v}")


── Chunk 1 content ─────────────────────────
When you see a photo like this, you can immediately tell that it's a picture of a bumblebee. How it is that biological nervous systems can do this so effortlessly is one of the great mysteries of neuroscience. But artificial neural networks take a different approach: learning by example. A neural network is made of layers of neurons. Each neuron holds a number between zero and one. The numbers in the first layer are the inputs — for an image, each pixel maps to one neuron. The last layer holds the output — one neuron per possible category (e.g. cat, dog, bumblebee). Layers in between are called hidden layers. Their neurons detect patterns. The activation of a neuron is determined by a weighted sum of all neurons in the previous layer. We add a bias — a number that shifts when the neuron starts firing — then squash the result through a sigmoid. Learning means finding weights and biases that make the network classify training examples correctl

In [13]:
# ── Ask the LLM about the video ─────────────────────────────────────────
yt_texts = [d.page_content for d in yt_docs]
yt_vecs  = np.array(embeddings.embed_documents(yt_texts), dtype="float32")
yt_index = faiss.IndexFlatL2(yt_vecs.shape[1])
yt_index.add(yt_vecs)

def yt_qa(question: str, k: int = 2) -> str:
    q_vec = np.array(embeddings.embed_query(question), dtype="float32").reshape(1, -1)
    _, idx = yt_index.search(q_vec, k)
    context = "\n\n".join(yt_texts[i] for i in idx[0])
    prompt = ChatPromptTemplate.from_template(
        "Answer based only on this video transcript.\n\n{ctx}\n\nQuestion: {q}"
    )
    return llm.invoke(prompt.format_messages(ctx=context, q=question)).content

q = "What is gradient descent and how does it help a neural network learn?"
print(f"Q: {q}")
print(f"A: {yt_qa(q)}")


Q: What is gradient descent and how does it help a neural network learn?


A: According to the transcript, gradient descent is an algorithm that helps a neural network learn by telling it which direction to nudge each weight and bias to reduce the cost. The cost is measured by a cost function, which is the sum of squared differences between predicted and true outputs. Gradient descent is used to minimize the cost, which is the goal of learning in a neural network.


---
## 4. JSON Dataset Loader
A rich QA dataset with metadata — the gold-standard format for RAG evaluation and fine-tuning.


In [14]:
import json

# ── Define and save a rich JSON dataset ─────────────────────────────────
qa_dataset = [
    {
        "id": "rag_001",
        "question": "What is Retrieval-Augmented Generation?",
        "ground_truth": "RAG is a technique that combines a retrieval system with an LLM generator. "
                        "The retriever fetches relevant documents; the LLM conditions its answer on them, "
                        "reducing hallucination and keeping knowledge current.",
        "topic": "RAG Basics",
        "difficulty": "beginner",
        "tags": ["RAG", "retrieval", "LLM"],
        "source": "synthetic",
        "metadata": {"created": "2025-01-01", "version": "1.0"}
    },
    {
        "id": "rag_002",
        "question": "What is the difference between dense and sparse retrieval?",
        "ground_truth": "Dense retrieval uses neural embeddings (e.g. all-MiniLM-L6-v2) and cosine similarity "
                        "to find semantically similar documents. Sparse retrieval uses term-frequency methods "
                        "like BM25 to match keyword occurrences. Hybrid retrieval combines both.",
        "topic": "Retrieval",
        "difficulty": "intermediate",
        "tags": ["BM25", "dense", "sparse", "hybrid"],
        "source": "synthetic",
        "metadata": {"created": "2025-01-01", "version": "1.0"}
    },
    {
        "id": "rag_003",
        "question": "What is HyDE (Hypothetical Document Embeddings)?",
        "ground_truth": "HyDE generates a hypothetical answer to the query using an LLM, embeds that answer, "
                        "and uses the embedding to search the vector store. This bridges the vocabulary gap "
                        "between short queries and longer documents.",
        "topic": "Advanced RAG",
        "difficulty": "intermediate",
        "tags": ["HyDE", "query expansion", "embeddings"],
        "source": "synthetic",
        "metadata": {"created": "2025-01-02", "version": "1.0"}
    },
    {
        "id": "rag_004",
        "question": "How does a cross-encoder reranker improve RAG?",
        "ground_truth": "A cross-encoder takes (query, document) pairs and scores their relevance jointly "
                        "rather than independently. After an initial retrieval of 20 candidates, the reranker "
                        "rescores all pairs and keeps only the top-5, significantly improving precision.",
        "topic": "Advanced RAG",
        "difficulty": "intermediate",
        "tags": ["reranker", "cross-encoder", "retrieval"],
        "source": "synthetic",
        "metadata": {"created": "2025-01-02", "version": "1.0"}
    },
    {
        "id": "rag_005",
        "question": "What is Self-RAG?",
        "ground_truth": "Self-RAG is a LangGraph workflow where the LLM grades retrieved documents for "
                        "relevance before generating. If documents are irrelevant, it refines the query "
                        "and re-retrieves. After generation, it grades the answer for hallucination.",
        "topic": "Advanced RAG",
        "difficulty": "advanced",
        "tags": ["Self-RAG", "LangGraph", "grading"],
        "source": "synthetic",
        "metadata": {"created": "2025-01-03", "version": "1.0"}
    },
    {
        "id": "rag_006",
        "question": "What is RAPTOR?",
        "ground_truth": "RAPTOR builds a multi-level summary tree: leaf nodes are document chunks, "
                        "which are clustered and summarised into mid-level nodes, then a final meta-summary. "
                        "All levels are indexed, enabling retrieval at multiple granularities.",
        "topic": "Advanced RAG",
        "difficulty": "advanced",
        "tags": ["RAPTOR", "summarization", "tree"],
        "source": "synthetic",
        "metadata": {"created": "2025-01-03", "version": "1.0"}
    },
    {
        "id": "rag_007",
        "question": "What is CRAG (Corrective RAG)?",
        "ground_truth": "CRAG grades retrieved documents before generation. If documents are 'correct' it "
                        "generates normally. If 'incorrect' it falls back to a web search. If 'ambiguous' "
                        "it combines both. This self-corrects retrieval failures.",
        "topic": "Advanced RAG",
        "difficulty": "advanced",
        "tags": ["CRAG", "corrective", "web search"],
        "source": "synthetic",
        "metadata": {"created": "2025-01-04", "version": "1.0"}
    },
    {
        "id": "rag_008",
        "question": "What is RRF (Reciprocal Rank Fusion)?",
        "ground_truth": "RRF merges ranked lists from multiple retrievers (e.g. dense + BM25) using "
                        "score = 1/(k + rank) where k=60. Higher-ranked results get higher scores. "
                        "Scores are summed across retrievers to produce a final merged ranking.",
        "topic": "Retrieval",
        "difficulty": "intermediate",
        "tags": ["RRF", "hybrid", "fusion"],
        "source": "synthetic",
        "metadata": {"created": "2025-01-04", "version": "1.0"}
    },
    {
        "id": "rag_009",
        "question": "What metrics does RAGAS use to evaluate RAG?",
        "ground_truth": "RAGAS measures: Faithfulness (are answers grounded in context?), "
                        "Answer Relevancy (does the answer address the question?), "
                        "Context Recall (was all needed info retrieved?), and "
                        "Context Precision (were retrieved chunks relevant?).",
        "topic": "Evaluation",
        "difficulty": "intermediate",
        "tags": ["RAGAS", "evaluation", "faithfulness"],
        "source": "synthetic",
        "metadata": {"created": "2025-01-05", "version": "1.0"}
    },
    {
        "id": "rag_010",
        "question": "How does semantic caching reduce LLM costs?",
        "ground_truth": "Semantic caching stores (query_embedding, answer) pairs. For each new query, "
                        "it checks cosine similarity against cached queries. If similarity >= threshold "
                        "(e.g. 0.92), it returns the cached answer without calling the LLM, saving cost.",
        "topic": "Production",
        "difficulty": "intermediate",
        "tags": ["cache", "semantic", "cost"],
        "source": "synthetic",
        "metadata": {"created": "2025-01-05", "version": "1.0"}
    },
    {
        "id": "rag_011",
        "question": "What is hallucination in LLM outputs?",
        "ground_truth": "Hallucination is when an LLM generates facts not present in the retrieved context. "
                        "It sounds confident but is made up. NLI cross-encoders detect this by scoring "
                        "P(entailment) between the context and the answer.",
        "topic": "Evaluation",
        "difficulty": "beginner",
        "tags": ["hallucination", "NLI", "detection"],
        "source": "synthetic",
        "metadata": {"created": "2025-01-06", "version": "1.0"}
    },
    {
        "id": "rag_012",
        "question": "What is parent-child chunking?",
        "ground_truth": "Parent-child chunking indexes small child chunks (300 chars) for precise retrieval "
                        "but returns the larger parent chunk (1500 chars) to the LLM for full context. "
                        "This gives precise matching with rich generation context.",
        "topic": "Ingestion",
        "difficulty": "intermediate",
        "tags": ["chunking", "parent-child", "retrieval"],
        "source": "synthetic",
        "metadata": {"created": "2025-01-06", "version": "1.0"}
    },
]

JSON_PATH = os.path.join(DS_DIR, 'rag_qa_dataset.json')
with open(JSON_PATH, 'w', encoding='utf-8') as f:
    json.dump(qa_dataset, f, indent=2)

print(f"✓ Saved JSON dataset: {JSON_PATH}")
print(f"✓ Total QA pairs: {len(qa_dataset)}")


✓ Saved JSON dataset: /home/saurabh/projects/personal_project/advance-rag/notebooks/../datasets/rag_qa_dataset.json
✓ Total QA pairs: 12


In [15]:
# ── Load JSON as LangChain Documents ────────────────────────────────────
def load_json_as_documents(json_path: str, text_key: str = "ground_truth") -> list[Document]:
    with open(json_path, encoding='utf-8') as f:
        data = json.load(f)
    docs = []
    for item in data:
        page_content = (
            f"Q: {item['question']}\n"
            f"A: {item[text_key]}"
        )
        metadata = {
            "id":         item["id"],
            "topic":      item["topic"],
            "difficulty": item["difficulty"],
            "tags":       ", ".join(item["tags"]),
            "source":     "json:" + item["source"],
        }
        docs.append(Document(page_content=page_content, metadata=metadata))
    return docs

json_docs = load_json_as_documents(JSON_PATH)
print(f"✓ Loaded {len(json_docs)} documents from JSON")
print()
print("── Sample document ─────────────────────────")
print(json_docs[0].page_content)
print()
print("── Metadata ────────────────────────────────")
for k, v in json_docs[0].metadata.items():
    print(f"  {k}: {v}")


✓ Loaded 12 documents from JSON

── Sample document ─────────────────────────
Q: What is Retrieval-Augmented Generation?
A: RAG is a technique that combines a retrieval system with an LLM generator. The retriever fetches relevant documents; the LLM conditions its answer on them, reducing hallucination and keeping knowledge current.

── Metadata ────────────────────────────────
  id: rag_001
  topic: RAG Basics
  difficulty: beginner
  tags: RAG, retrieval, LLM
  source: json:synthetic


In [16]:
# ── Dataset statistics ───────────────────────────────────────────────────
from collections import Counter

with open(JSON_PATH, encoding='utf-8') as f:
    data = json.load(f)

topics      = Counter(item['topic'] for item in data)
difficulties = Counter(item['difficulty'] for item in data)
all_tags     = Counter(tag for item in data for tag in item['tags'])

print("── Dataset overview ────────────────────────")
print(f"  Total QA pairs : {len(data)}")
print()
print("  By topic:")
for topic, count in sorted(topics.items(), key=lambda x: -x[1]):
    print(f"    {topic:<25} {count} questions")
print()
print("  By difficulty:")
for diff, count in sorted(difficulties.items()):
    print(f"    {diff:<12} {count} questions")
print()
print("  Top tags:")
for tag, count in all_tags.most_common(8):
    print(f"    #{tag:<20} {count}x")


── Dataset overview ────────────────────────
  Total QA pairs : 12

  By topic:
    Advanced RAG              5 questions
    Retrieval                 2 questions
    Evaluation                2 questions
    RAG Basics                1 questions
    Production                1 questions
    Ingestion                 1 questions

  By difficulty:
    advanced     3 questions
    beginner     2 questions
    intermediate 7 questions

  Top tags:
    #retrieval            3x
    #hybrid               2x
    #RAG                  1x
    #LLM                  1x
    #BM25                 1x
    #dense                1x
    #sparse               1x
    #HyDE                 1x


In [17]:
# ── Semantic search over JSON dataset ───────────────────────────────────
json_texts = [d.page_content for d in json_docs]
json_vecs  = np.array(embeddings.embed_documents(json_texts), dtype="float32")
json_index = faiss.IndexFlatL2(json_vecs.shape[1])
json_index.add(json_vecs)

def json_search(query: str, k: int = 3):
    q_vec = np.array(embeddings.embed_query(query), dtype="float32").reshape(1, -1)
    _, idx = json_index.search(q_vec, k)
    return [json_docs[i] for i in idx[0]]

query = "how does retrieval work in advanced rag systems?"
results = json_search(query)
print(f"Query: '{query}'")
print(f"Top {len(results)} matches:")
for i, doc in enumerate(results, 1):
    m = doc.metadata
    print(f"\n  {i}. [{m['id']}] {m['topic']} ({m['difficulty']})")
    print(f"     Tags: {m['tags']}")
    print(f"     {doc.page_content[:120]}...")


Query: 'how does retrieval work in advanced rag systems?'
Top 3 matches:

  1. [rag_001] RAG Basics (beginner)
     Tags: RAG, retrieval, LLM
     Q: What is Retrieval-Augmented Generation?
A: RAG is a technique that combines a retrieval system with an LLM generator....

  2. [rag_009] Evaluation (intermediate)
     Tags: RAGAS, evaluation, faithfulness
     Q: What metrics does RAGAS use to evaluate RAG?
A: RAGAS measures: Faithfulness (are answers grounded in context?), Answ...

  3. [rag_005] Advanced RAG (advanced)
     Tags: Self-RAG, LangGraph, grading
     Q: What is Self-RAG?
A: Self-RAG is a LangGraph workflow where the LLM grades retrieved documents for relevance before g...


---
## 5. Summary — All Loaders at a Glance


In [18]:
print("=" * 60)
print("DATA LOADER SUMMARY")
print("=" * 60)
print(f"{'Source':<20} {'Documents':>10} {'Use Case'}")
print("-" * 60)
print(f"{'SQLite DB':<20} {len(all_docs):>10}   Structured records, filterable by SQL")
print(f"{'PDF (Attention)':<20} {len(pdf_chunks):>10}   Research papers, product docs")
print(f"{'YouTube Transcript':<20} {len(yt_docs):>10}   Video content, lectures, podcasts")
print(f"{'JSON QA Dataset':<20} {len(json_docs):>10}   Eval sets, knowledge bases, FAQs")
print("=" * 60)
print()
print("Files saved to datasets/:")
for fname in ['ai_papers.db', 'rag_qa_dataset.json', 'yt_transcript_sample.json']:
    path = os.path.join(DS_DIR, fname)
    size = os.path.getsize(path) if os.path.exists(path) else 0
    print(f"  {fname:<35} {size:>8,} bytes")


DATA LOADER SUMMARY
Source                Documents Use Case
------------------------------------------------------------
SQLite DB                    10   Structured records, filterable by SQL
PDF (Attention)              51   Research papers, product docs
YouTube Transcript            4   Video content, lectures, podcasts
JSON QA Dataset              12   Eval sets, knowledge bases, FAQs

Files saved to datasets/:
  ai_papers.db                           8,192 bytes
  rag_qa_dataset.json                    6,894 bytes
  yt_transcript_sample.json              5,921 bytes
